# MacroFit — Google Colab trial notebook

Run one cell at a time from top to bottom. Change only the values in the clearly marked input cells, then rerun the cells below them to see the impact. The notebook loads the project and JSON datasets from Google Drive.

## 1. Mount Google Drive
Colab will ask you to authorize access to your own Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Point to the project folder in Drive
Upload the complete `PE6201_Pavan_Nagur` repository folder to **My Drive**, or change the path below to its actual location.

In [ ]:
from pathlib import Path

PROJECT_FOLDER = Path('/content/drive/MyDrive/PE6201_Pavan_Nagur')  # CHANGE THIS IF NEEDED
required_files = [
    PROJECT_FOLDER / 'core.py',
    PROJECT_FOLDER / 'data' / 'menu.json',
    PROJECT_FOLDER / 'data' / 'nutrition.json',
    PROJECT_FOLDER / 'data' / 'meal_rules.json',
]

print('Project folder:', PROJECT_FOLDER)
for path in required_files:
    print('FOUND' if path.exists() else 'MISSING', '-', path.name)

if not all(path.exists() for path in required_files):
    raise FileNotFoundError('Update PROJECT_FOLDER so it points to the full repository folder in Google Drive.')

## 3. Load the two JSON datasets separately

In [ ]:
import json
import pandas as pd

menu_rows = json.loads((PROJECT_FOLDER / 'data' / 'menu.json').read_text())
menu_df = pd.DataFrame(menu_rows)
print(f'Menu dataset: {len(menu_df)} items across {menu_df.shop.nunique()} shops')
display(menu_df.head())

In [ ]:
nutrition_rows = json.loads((PROJECT_FOLDER / 'data' / 'nutrition.json').read_text())
nutrition_df = pd.DataFrame(nutrition_rows)
print(f'Nutrition dataset: {len(nutrition_df)} nutrition records')
display(nutrition_df.head())

## 4. Join menu and nutrition data by `item_id`
This cell exposes missing or duplicated mappings—the silent-error area highlighted in the project feedback.

In [ ]:
joined_df = menu_df.merge(nutrition_df, on='item_id', how='outer', indicator=True, validate='one_to_one')
print(joined_df['_merge'].value_counts())
assert (joined_df['_merge'] == 'both').all(), 'Some menu and nutrition rows did not match.'
display(joined_df.drop(columns='_merge').head(10))

## 5. Load the tested Python functions from Drive
Rerun this cell after changing `core.py` so Colab reloads the new version.

In [ ]:
import importlib
import sys

project_path = str(PROJECT_FOLDER)
if project_path not in sys.path:
    sys.path.insert(0, project_path)

import core
importlib.reload(core)

menu = core.load_menu()
meal_rules = core.load_meal_rules()
print('Loaded', len(menu), 'joined menu items')
print('Cheapest valid three-meal day: S$', core.minimum_three_meal_budget(menu, meal_rules), sep='')

## 6. Change the user profile
These are the only inputs used by the deterministic calorie and macro calculation.

In [ ]:
AGE = 24
WEIGHT_KG = 72
HEIGHT_CM = 175
CALCULATION_SEX = 'male'          # 'male' or 'female'
ACTIVITY_LEVEL = 'moderately_active'  # sedentary, lightly_active, moderately_active, very_active, extra_active
GOAL = 'maintain'                 # lose, maintain, gain

print('Profile inputs saved. Run the next cell to calculate targets.')

## 7. Calculate daily targets with Python—no AI

In [ ]:
targets = core.calculate_daily_targets(
    AGE, WEIGHT_KG, HEIGHT_CM, CALCULATION_SEX, ACTIVITY_LEVEL, GOAL
)
display(pd.DataFrame([targets]).T.rename(columns={0: 'value'}))

## 8. Set budget and meal-specific food preferences
The structured values are hard constraints. You can also test the same meaning in natural language.

In [ ]:
BUDGET_SGD = 30.00
FOOD_MOOD = 'Maybe veg for lunch and chicken for dinner'

# Allowed values for each meal: 'any', 'vegetarian', 'chicken'
EXPLICIT_MEAL_PREFERENCES = {
    'breakfast': 'any',
    'lunch': 'any',
    'dinner': 'any',
}

meal_preferences = core.parse_meal_preferences(FOOD_MOOD, EXPLICIT_MEAL_PREFERENCES)
print('Hard constraints understood by the optimizer:', meal_preferences)

## 9. Run the deterministic three-meal optimizer
The optimizer—not the language model—enforces budget, meal type, food constraints, and the 90% nutrition threshold.

In [ ]:
solution = core.optimize_daily_meal_plan(
    menu=menu,
    budget=BUDGET_SGD,
    calories_kcal=targets['estimated_daily_calories_kcal'],
    protein_g=targets['protein_target_g'],
    carbs_g=targets['carb_target_g'],
    rules=meal_rules,
    meal_preferences=meal_preferences,
)

print('Plan available:', solution['available'])
print('Plans evaluated:', solution['evaluated_count'])
print('Plans meeting all constraints:', solution['feasible_count'])
print('Minimum target-matching budget:', solution.get('minimum_target_budget_sgd'))
if not solution['available']:
    print('Reason:', solution['message'])

## 10. Inspect the chosen plan and target coverage

In [ ]:
if solution['available']:
    plan = solution['optimizer_pick']
    schedule_rows = []
    for entry in plan['schedule']:
        item = entry['item']
        schedule_rows.append({
            'meal': entry['meal'],
            'time': entry['eat_at'],
            'item': item['item'],
            'cuisine': item['cuisine'],
            'servings': entry['quantity'],
            'cost_sgd': round(item['price_sgd'] * entry['quantity'], 2),
            'kcal': round(item['kcal'] * entry['quantity']),
        })
    display(pd.DataFrame(schedule_rows))
    display(pd.DataFrame([plan['totals']]))
    print('Calorie coverage:', round(plan['totals']['kcal'] / targets['estimated_daily_calories_kcal'] * 100, 1), '%')
    print('Protein coverage:', round(plan['totals']['protein_g'] / targets['protein_target_g'] * 100, 1), '%')
    print('Carbohydrate coverage:', round(plan['totals']['carbs_g'] / targets['carb_target_g'] * 100, 1), '%')
else:
    print(solution['message'])

## 11. Test the impact of a lower budget
This reproduces the earlier S$18.10 case. It should return no plan instead of presenting a 1,900 kcal plan as sufficient.

In [ ]:
low_budget_solution = core.optimize_daily_meal_plan(
    menu, 18.10,
    targets['estimated_daily_calories_kcal'],
    targets['protein_target_g'],
    targets['carb_target_g'],
    rules=meal_rules,
    meal_preferences=meal_preferences,
)
print('Plan available:', low_budget_solution['available'])
print(low_budget_solution['message'])
print('Calorie shortfall to 90% threshold:', low_budget_solution.get('calorie_shortfall_kcal'))

## 12. Compare multiple budgets side by side

In [ ]:
comparison = []
for trial_budget in [18.10, 22, 25, 30, 35]:
    trial = core.optimize_daily_meal_plan(
        menu, trial_budget,
        targets['estimated_daily_calories_kcal'],
        targets['protein_target_g'],
        targets['carb_target_g'],
        rules=meal_rules,
        meal_preferences=meal_preferences,
    )
    pick = trial.get('optimizer_pick')
    comparison.append({
        'budget_sgd': trial_budget,
        'available': trial['available'],
        'minimum_required_sgd': trial.get('minimum_target_budget_sgd'),
        'plan_cost_sgd': pick['totals']['price_sgd'] if pick else None,
        'plan_kcal': pick['totals']['kcal'] if pick else None,
        'message': trial['message'],
    })
display(pd.DataFrame(comparison))

## 13. Inspect the meal-planning tool descriptor
This shows the exact contract the assistant must follow.

In [ ]:
descriptors = json.loads((PROJECT_FOLDER / 'config' / 'tool_descriptors.json').read_text())
meal_plan_descriptor = next(tool for tool in descriptors['tools'] if tool['name'] == 'create_daily_meal_plan')
print(json.dumps(meal_plan_descriptor, indent=2))

## 14. Optional: let OpenRouter rank the valid shortlist
The key is entered with hidden input and is kept only in the current Colab runtime. It is not written to Drive or the notebook. Skip this cell if you only want the deterministic optimizer.

In [ ]:
from getpass import getpass
import os

if not solution['available']:
    print('No valid shortlist is available. Increase the budget and rerun the optimizer first.')
else:
    os.environ['OPENROUTER_API_KEY'] = getpass('OpenRouter API key (hidden): ')
    os.environ['OPENROUTER_MODEL'] = 'qwen/qwen3.8-flash'
    import app
    importlib.reload(app)
    model_ranking = app.openrouter_rank(solution['shortlist'], FOOD_MOOD)
    print(json.dumps(model_ranking, indent=2))

## 15. Simulate what was actually eaten
Change an item ID or quantity to simulate a deviation, then rerun this cell. Nutrition values are always loaded from the dataset.

In [ ]:
if solution['available']:
    actual_items = [
        {
            'meal': entry['meal'],
            'item_id': entry['item']['item_id'],
            'quantity': entry['quantity'],
        }
        for entry in solution['optimizer_pick']['schedule']
    ]
    print('Edit this list to test a deviation:')
    print(json.dumps(actual_items, indent=2))
else:
    actual_items = []
    print('Create an available plan first.')

In [ ]:
if actual_items:
    actual_analysis = core.analyze_actual_intake(
        menu,
        actual_items,
        {
            'calories_kcal': targets['estimated_daily_calories_kcal'],
            'protein_g': targets['protein_target_g'],
            'carbs_g': targets['carb_target_g'],
            'fat_g': targets['fat_target_g'],
        },
        BUDGET_SGD,
    )
    print(json.dumps(actual_analysis, indent=2))

## Suggested experiments

1. Change activity level and observe the calorie estimate.
2. Compare S$18.10, S$25, and S$30 budgets.
3. Try `veg for lunch and chicken for dinner`.
4. Set all three explicit meal preferences to `vegetarian`.
5. Change an actual item or serving quantity and compare the completed-day analysis.
6. Compare the optimizer pick with the optional OpenRouter pick from the same shortlist.